#  Open Food Facts Canada - Data Analysis

Exploratory Data Analysis for Canadian food products

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to database
conn = duckdb.connect('off_canada.duckdb')
print('Connected to database!')

## 📊 1. Top Brands Analysis

In [ ]:
top_brands = conn.execute("""
    SELECT 
        brands,
        COUNT(*) as product_count
    FROM silver_products
    WHERE brands IS NOT NULL AND brands != ''
    GROUP BY brands
    ORDER BY product_count DESC
    LIMIT 10
""").fetchdf()

top_brands

In [ ]:
# Visualize top brands
plt.figure(figsize=(12, 6))
sns.barplot(data=top_brands, x='product_count', y='brands', palette='viridis')
plt.title('Top 10 Brands by Product Count', fontsize=16)
plt.xlabel('Number of Products')
plt.ylabel('Brand')
plt.tight_layout()
plt.show()

##  2. Sugar Content Analysis

In [ ]:
high_sugar = conn.execute("""
    SELECT 
        brands,
        product_name,
        sugars_100g,
        main_category
    FROM silver_products
    WHERE sugars_100g IS NOT NULL
    ORDER BY sugars_100g DESC
    LIMIT 15
""").fetchdf()

high_sugar

In [ ]:
# Sugar distribution
sugar_dist = conn.execute("""
    SELECT 
        CASE 
            WHEN sugars_100g = 0 THEN 'No Sugar'
            WHEN sugars_100g <= 5 THEN 'Low (≤5g)'
            WHEN sugars_100g <= 15 THEN 'Medium (5-15g)'
            ELSE 'High (>15g)'
        END as sugar_level,
        COUNT(*) as count
    FROM silver_products
    WHERE sugars_100g IS NOT NULL
    GROUP BY sugar_level
""").fetchdf()

plt.figure(figsize=(8, 8))
plt.pie(sugar_dist['count'], labels=sugar_dist['sugar_level'], autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Sugar Content in Products', fontsize=14)
plt.show()

## 3. Nutri-Score Distribution

In [ ]:
nutriscore = conn.execute("""
    SELECT 
        nutriscore_grade,
        COUNT(*) as count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) as percentage
    FROM silver_products
    WHERE nutriscore_grade IN ('a', 'b', 'c', 'd', 'e')
    GROUP BY nutriscore_grade
    ORDER BY nutriscore_grade
""").fetchdf()

nutriscore

In [ ]:
# Nutri-Score visualization
colors = {'a': 'green', 'b': 'lightgreen', 'c': 'yellow', 'd': 'orange', 'e': 'red'}
plt.figure(figsize=(10, 6))
bars = plt.bar(nutriscore['nutriscore_grade'], nutriscore['count'], 
               color=[colors.get(g, 'gray') for g in nutriscore['nutriscore_grade']])
plt.title('Nutri-Score Distribution', fontsize=16)
plt.xlabel('Nutri-Score Grade')
plt.ylabel('Number of Products')
for bar, pct in zip(bars, nutriscore['percentage']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
             f'{pct}%', ha='center', va='bottom')
plt.show()

## 🍽️ 4. Nutrition by Category

In [ ]:
category_nutrition = conn.execute("""
    SELECT 
        main_category,
        COUNT(*) as product_count,
        ROUND(AVG(energy_kcal_100g), 2) as avg_energy,
        ROUND(AVG(sugars_100g), 2) as avg_sugar,
        ROUND(AVG(proteins_100g), 2) as avg_protein
    FROM silver_products
    WHERE main_category IS NOT NULL
    GROUP BY main_category
    HAVING COUNT(*) >= 10
    ORDER BY product_count DESC
    LIMIT 10
""").fetchdf()

category_nutrition

In [ ]:
# Category comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sns.barplot(data=category_nutrition, x='avg_energy', y='main_category', ax=axes[0], palette='Reds')
axes[0].set_title('Average Energy (kcal)')

sns.barplot(data=category_nutrition, x='avg_sugar', y='main_category', ax=axes[1], palette='Blues')
axes[1].set_title('Average Sugar (g)')

sns.barplot(data=category_nutrition, x='avg_protein', y='main_category', ax=axes[2], palette='Greens')
axes[2].set_title('Average Protein (g)')

plt.tight_layout()
plt.show()

##  5. Correlation Analysis

In [ ]:
# Correlation between nutrients
corr_data = conn.execute("""
    SELECT 
        energy_kcal_100g,
        proteins_100g,
        carbohydrates_100g,
        fat_100g,
        sugars_100g,
        sodium_100g
    FROM silver_products
    WHERE energy_kcal_100g IS NOT NULL
""").fetchdf()

correlation = corr_data.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5)
plt.title('Nutrition Correlation Matrix', fontsize=16)
plt.show()

In [ ]:
# Close connection
conn.close()
print('Analysis complete!')